In [1]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

Each row has the generated question, the original FAQ answer, and the answer produced by the RAG pipeline.

This is offline evaluation. We can do it because our test dataset came from FAQ records. We know the original answer for every generated question.

In production, we usually don't have that original answer for real user questions. There we can still use an LLM judge. The prompt has to judge only the question and the generated answer. In this lesson, we use the stronger offline setup.



A->Q->A' evaluation
We'll compare the RAG answer with the original answer from the FAQ. This checks if the RAG pipeline is producing answers that match the ground truth.

First, define the output format:

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

The judge returns two fields. The score gives us a metric we can aggregate. The reasoning explains the score, which helps when we look at bad examples.

First, write the judge instructions. This tells the judge what to compare and how to assign the score.



In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [4]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [10]:
rec = answers[0]

In [11]:
rec

{'question': 'I just found this course, is it too late for me to join, or can I still sign up?',
 'answer_llm': 'Yes, you can still join. You don’t need a confirmation email or prior registration—just start learning and submit homework while the form is open. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [13]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

Call the judge:

In [14]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core message: it is still possible to join, and to receive a certificate the student must submit the project while submissions are still open. The extra details about no confirmation email or prior registration do not change the main meaning and are compatible with the ground truth.', score='good')

In [15]:
calc_price(usage)

{'input_cost': 0.00024150000000000002,
 'output_cost': 0.000324,
 'total_cost': 0.0005655}

Now put the same logic into a function:

In [16]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

Test it on the same record:

In [17]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key point from the ground truth: it is still possible to join, and certificate eligibility depends on submitting the project while submissions are still open. The extra mention about confirmation email/prior registration is not in the original, but it does not conflict with the main answer and does not change the core meaning.', score='good')

Running the judge

Run the evaluation on all answers:

In [18]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

Use the same parallel processing helper:

In [19]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/565 [00:00<?, ?it/s]

Split the results:

In [20]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

Create a dataframe:

In [21]:
df_eval = pd.DataFrame(evaluations)

In [22]:
calc_total_price(usages)

0.3989347499999999

Check the results:

In [23]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 536/565 = 94.87%


Look at the "bad" cases to understand what went wrong:

In [24]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
2,What do I need to do to qualify for the certif...,74eb249bbf,bad,The AI answer adds specific certificate requir...
3,Can I submit my final project after the submis...,74eb249bbf,bad,"The ground truth says yes, late submission is ..."
32,Is the Capstone project the only requirement f...,9f689c185f,bad,The ground truth says the Capstone project is ...
33,What happens if I miss some homework in this c...,9f689c185f,bad,The AI answer does not address the question an...
43,Is there a date for the upcoming run of the co...,bd31146b0e,bad,The ground truth provides a specific date/time...


These rows are often the most useful part of the evaluation. They can show that search retrieved the wrong document. They can also show that the answer is too generic. Sometimes the RAG pipeline says that it doesn't know even though the FAQ had the answer.



Evaluating the judge
The judge can be wrong. It may rate an answer as good even though search failed to retrieve the right document. In that case the judge is too lenient. Make the instructions stricter and re-run the evaluation.

To evaluate the judge, you need to look at the results yourself. Sample some good and bad cases, read the judge reasoning, and check whether you agree with the verdict. You cannot use another judge to evaluate the judge. This is manual work, but it is necessary.

A practical approach is to build a simple application with Streamlit. Show each question, the original answer, the generated answer, and the judge verdict side by side. Then mark each verdict as correct or incorrect and use that feedback to adjust the judge instructions. This is a lot of trial and error, but it makes the evaluation framework more reliable.



Saving the results
Save the judged answers:

In [25]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)